In [0]:
DEBUG = True

DATADIR = '/datascope/subaru/data/datastore'
RUN = 'u_price_dobos-20260327_20260327T185911Z'
RUNDIR = f'u/price/dobos-20260327/20260327T185911Z'
CONFIGRUN = 'PFS_raw_pfsConfig'
CONFIGRUNDIR = 'PFS/raw/pfsConfig'

CATID = 10092
OBJID = 0x00000002000055dc
VISIT = 122806

In [0]:
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
if DEBUG and 'debugpy' not in globals():
    import debugpy
    debugpy.listen(5683)
    print("Waiting for debugger to attach...")

In [0]:
from pfs.ga.pfsspec.survey.pfs.datamodel import PfsCalibrated, PfsCalibratedLsf, PfsConfig
from pfs.ga.pfsspec.survey.repo import FileSystemRepo
from pfs.ga.pfsspec.survey.pfs import PfsGen3Repo, PfsGen3FileSystemConfig

In [0]:
repo = PfsGen3Repo(FileSystemRepo, PfsGen3FileSystemConfig)
repo.set_variable('datadir', DATADIR)
repo.set_variable('rundir', RUNDIR)
repo.set_variable('configrundir', CONFIGRUNDIR)

In [0]:
filename, identity = repo.locate_product(PfsConfig, visit=VISIT)
filename, identity

In [0]:
config, _, _ = repo.load_product(PfsConfig, filename=filename)
config

In [0]:
filename, identity = repo.locate_product(PfsCalibratedLsf, visit=VISIT, catid=CATID)
filename, identity

In [0]:
lsf_all, _, _ = repo.load_product(PfsCalibratedLsf, filename=filename)
len(lsf_all)

In [0]:
filename, identity = repo.locate_product(PfsCalibrated, visit=VISIT, catid=CATID)
filename, identity

In [0]:
spec_all, _, _ = repo.load_product(PfsCalibrated, filename=filename)
len(spec_all)

In [0]:
# Indexed by Target
lsf_all.keys()

In [0]:
target, lsf = next(iter(lsf_all.items()))
target, type(lsf)

In [0]:
lsf.__dict__.keys()

In [0]:
# The LSFs for the three arms
lsf.lsfList

In [0]:
# These are likely the pixel limits for each LSF
lsf.minIndex, lsf.maxIndex

In [0]:
lsf.weights

In [0]:
# This is the same as the rebinned spectra if pfsCalibrated.flux
# Not the original pixels in fluxTable
lsf.length

In [0]:
lsf.interpolator

In [0]:
lsf.lsfList[0]

In [0]:
lsf.lsfList[0].__dict__.keys()

In [0]:
lsf.lsfList[0].length

In [0]:
lsf.lsfList[0].width,  lsf.lsfList[0].kernel, lsf.lsfList[0].norm

In [0]:
plt.plot(lsf.lsfList[0].kernel.values)
plt.plot(lsf.lsfList[1].kernel.values)
plt.plot(lsf.lsfList[2].kernel.values)

In [0]:
len(lsf.lsfList)

In [0]:
next(iter(spec_all.values())).wavelength

In [0]:
# Try to figure out how to tell the spectrograph arm from the LSF
lsf_min_index = np.array([lsf.minIndex for target, lsf in lsf_all.items()])
lsf_max_index = np.array([lsf.maxIndex for target, lsf in lsf_all.items()])
lsf_min_wave = np.array([spec_all[target].wavelength[lsf.minIndex] for target, lsf in lsf_all.items()])
lsf_max_wave = np.array([spec_all[target].wavelength[lsf.maxIndex] for target, lsf in lsf_all.items()])
lsf_min_index.shape

In [0]:
np.unique(lsf_min_index.ravel())

In [0]:
np.unique(lsf_max_index.ravel())

In [0]:
np.unique(lsf_min_wave)

In [0]:
np.unique(lsf_max_wave)

In [0]:
# Get the width of the lsf for each target and each arm
lsf_width = defaultdict(dict)
lsf_spectrograph = defaultdict(dict)
for target, lsf in lsf_all.items():
    for i, lsf_arm in enumerate(lsf.lsfList):
        # This is just a rough way to determine which arm we are looking at
        # lsf.minIndex, lsf.maxIndex
        # (array([4201, 9033,   48]), array([ 8524, 13176,  3576]))
        if spec_all[target].wavelength[lsf.minIndex[i]] < 500:
            arm = 'b'
        elif spec_all[target].wavelength[lsf.maxIndex[i]] > 1000:
            arm = 'n'
        else:
            arm = 'mr'
        lsf_width[arm][target.objId] = lsf_arm.width

        # This is wrong, all values are 0
        # lsf_spectrograph[arm][target.objId] = spec_all[target].observations.spectrograph[0]

        # Instead, look up the spectrograph id in PfsConfig
        config_idx = np.where(config.objId == target.objId)[0][0]
        lsf_spectrograph[arm][target.objId] = config.spectrograph[config_idx]

In [0]:
fig, axes = plt.subplots(4, 3, figsize=(12, 12), sharey=True)

for j, spectrograph in enumerate(range(1, 5)):
    for i, arm in enumerate(['b', 'mr', 'n']):
        ax = axes[j, i]
        mask = np.array(list(lsf_spectrograph[arm].values())) == spectrograph
        values = np.array(list(lsf_width[arm].values()))
        hist, bins = np.histogram(values[mask], bins=20)
        ax.bar(bins[:-1], hist, width=np.diff(bins))
        # if j == 0:
        ax.set_title(f'Arm {arm}, spectrograph {spectrograph}')
        if j == 3:
            ax.set_xlabel('LSF Width')
        ax.set_ylabel('Number of Targets')

fig.tight_layout()

In [0]:
spec_all[next(iter(spec_all))].observations.spectrograph

In [0]:
fig, axes = plt.subplots(4, 3, figsize=(12, 12), sharey=True)

for j, spectrograph in enumerate(range(1, 5)):
    for i, arm in enumerate(['b', 'mr', 'n']):
        ax = axes[j, i]
        mask = np.array(list(lsf_spectrograph[arm].values())) == spectrograph
        values = np.array(list(lsf_width[arm].values()))
        hist, bins = np.histogram(values[mask] - np.median(values[mask]), bins=20)
        ax.bar(bins[:-1], hist, width=np.diff(bins))
        # if j == 0:
        ax.set_title(f'Arm {arm}, spectrograph {spectrograph}')
        if j == 3:
            ax.set_xlabel('LSF Width')
        ax.set_ylabel('Number of Targets')

fig.tight_layout()